joblib, regex, tqdm

In [4]:
from tokenizer import Tokenizer, train_bpe
import json
from pathlib import Path

class TokenizerData:
    """JSON adapter for a trained tokenizer's on-disk representation
    (tokenizer.json): vocab, merges, special_tokens, vocab_size.

    Byte strings round-trip through latin-1 text, which is a 1:1 mapping
    between byte values 0-255 and Unicode code points 0-255 -- unlike
    utf-8, it can encode/decode *any* byte sequence, including ones that
    aren't valid utf-8 (BPE merges routinely produce these).
    """

    def __init__(
        self,
        vocab: dict[int, bytes],
        merges: list[tuple[bytes, bytes]],
        special_tokens: list[str],
        vocab_size: int,
    ):
        self.vocab = vocab
        self.merges = merges
        self.special_tokens = special_tokens
        self.vocab_size = vocab_size

    def to_json(self) -> dict:
        return {
            "vocab_size": self.vocab_size,
            "special_tokens": self.special_tokens,
            "vocab": {str(idx): tok.decode("latin-1") for idx, tok in self.vocab.items()},
            "merges": [[a.decode("latin-1"), b.decode("latin-1")] for a, b in self.merges],
        }

    @classmethod
    def from_json(cls, data: dict) -> "TokenizerData":
        missing = {"vocab", "merges", "special_tokens", "vocab_size"} - data.keys()
        if missing:
            raise ValueError(f"tokenizer.json missing required key(s): {sorted(missing)}")
        vocab = {int(idx): tok.encode("latin-1") for idx, tok in data["vocab"].items()}
        merges = [(a.encode("latin-1"), b.encode("latin-1")) for a, b in data["merges"]]
        return cls(vocab, merges, data["special_tokens"], data["vocab_size"])

    def save(self, path: Path):
        path.write_text(json.dumps(self.to_json(), indent=2))

    @classmethod
    def load(cls, path: Path) -> "TokenizerData":
        return cls.from_json(json.loads(path.read_text()))


In [5]:
vocab, merges = train_bpe('/Users/oguz/Projects/launchpad/docs/odyssey.txt',
          vocab_size=1000,
          special_tokens=[]
          )

pretokenizing and building frequency map: 100%|██████████| 718k/718k [00:00<00:00, 14.7MB/s]
merging pairs: 100%|██████████| 744/744 [00:01<00:00, 375.36it/s]


In [10]:
adapter = TokenizerData(vocab=vocab,merges=merges,special_tokens=[],vocab_size=1000)

In [ ]:
adapter.from_json()